In [1]:
!pip install -r ../requirements.txt

In [2]:
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())

12.8
True


In [3]:
import sys
import importlib
sys.path.append("..")

# Import your modules first
import helper
import obelix
import algorithms.d3qn_per_agent_lstm as d3qn_per_agent
import configurations.config_p2_sub5_v2 as config

# Reload them to pick up any changes
importlib.reload(helper)
importlib.reload(obelix)
importlib.reload(d3qn_per_agent)
importlib.reload(config)

device used:  cuda
device used:  cuda


<module 'configurations.config_p2_sub5_v2' from 'c:\\Users\\hgoel\\MTech_Resources\\DRL\\Capstone_Project\\CS780-OBELIX\\run_scripts\\..\\configurations\\config_p2_sub5_v2.py'>

In [4]:
from helper import plotQuantity
from obelix import OBELIX
from algorithms.d3qn_per_agent_lstm import D3QN_PER
from configurations.config_p2_sub5_v2 import config

In [5]:
import sequenceReplayBuffer
print(sequenceReplayBuffer.__file__)
import inspect
print(inspect.getsource(sequenceReplayBuffer.SeqReplayBuffer.sample))

c:\Users\hgoel\MTech_Resources\DRL\Capstone_Project\CS780-OBELIX\run_scripts\..\sequenceReplayBuffer.py
    def sample(self, **kwargs):
        priorities = np.array(self.prioritiesOverEpisodes, dtype=np.float32)

        # clean priorities
        priorities = np.nan_to_num(priorities, nan=1e-6, posinf=1.0, neginf=1e-6)
        priorities = np.abs(priorities) + 1e-6

        probs = priorities ** self.alpha

        # clean probs
        probs = np.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)

        prob_sum = probs.sum()

        if prob_sum <= 0 or not np.isfinite(prob_sum):
            probs = np.ones(len(probs)) / len(probs)
        else:
            probs = probs / prob_sum

        batch_experiences = []
        batch_weights = []
        start_indices = []

        N = self.length()
        batch = min(self.batchSize, N)

        indices = np.random.choice(N, batch, replace=True, p=probs)

        updatedBeta = min(1.0, self.beta + self.beta_rate * kwargs['running_step'

In [6]:
print(config)

{'seed': 333, 'gamma': 0.999, 'bufferSize': 100000, 'batchSize': 32, 'optimizerFn': <class 'torch.optim.adam.Adam'>, 'optimizerLR': 0.0003, 'MAX_TRAIN_EPISODES': 250, 'MAX_EVAL_EPISODES': 5, 'updateFrequency': 150, 'explorationStrategyTrainFn': <function selectEpsilonGreedyAction at 0x000001C63263C040>, 'explorationStrategyEvalFn': <function selectGreedyAction at 0x000001C63261BF60>, 'max_steps': 1000, 'epochs': 20, 'epsilon': 0.58, 'eps_decay_strategy': [('exponential', {'s': 0, 'e': 249, 'ival': 1.0, 'fval': 0.005})], 'device': device(type='cuda'), 'delta': 0.99, 'tau': 0.005, 'alpha': 0.58, 'beta': 0.42, 'beta_rate': 0.0002, 'f_hDim': [512, 512, 256], 'lstm_hDim': 256, 'model_path': '../model_weights_phase2_sub5_v2', 'loss_fn': 'HuberLoss', 'seq_len': 30, 'burn_in': 15, 'minSamples': 2}


In [7]:
env = OBELIX(
    scaling_factor=5,
    arena_size=500,
    max_steps=1000,
    wall_obstacles=True,
    difficulty=2
)
d3qnPerwithLSTMAgent = D3QN_PER(env, config)

In [8]:
d3qnPerTrainRewardsList, d3qnPerTrainTimeList, d3qnPerEvalRewardsList, d3qnPerWallClockTimeList, d3qnPerTotalStepsList, d3qnPerFinalEvalReward = d3qnPerwithLSTMAgent.runD3QN_PER()

Episode 0: TR -8175.0 | ER -119716.4 | TT 8.73971176147461 | WC 49.640206813812256 | TS 1000
Episode 1: TR 1168.0 | ER -115728.4 | TT 57.55092978477478 | WC 96.45942378044128 | TS 962
Episode 2: TR -9784.0 | ER -119485.4 | TT 110.12890648841858 | WC 141.89801025390625 | TS 1000
Episode 3: TR -14173.0 | ER -980.8 | TT 154.40079283714294 | WC 198.3957600593567 | TS 1000
Episode 4: TR -974.0 | ER -120223.0 | TT 223.1890344619751 | WC 298.41889667510986 | TS 1000
Episode 5: TR -15976.0 | ER -991.6 | TT 321.1690912246704 | WC 398.4115617275238 | TS 1000
Episode 6: TR -67569.0 | ER -990.8 | TT 421.7813928127289 | WC 490.8977904319763 | TS 1000
Episode 7: TR -12975.0 | ER -991.2 | TT 503.0835816860199 | WC 565.8921239376068 | TS 1000
Episode 8: TR -16774.0 | ER -997.4 | TT 587.704042673111 | WC 663.1657168865204 | TS 1000
Episode 9: TR -5803.0 | ER -994.6 | TT 679.0280556678772 | WC 754.4904606342316 | TS 734
Episode 10: TR -72186.0 | ER -40871.8 | TT 777.8450794219971 | WC 853.04545378685 | 

AttributeError: 'numpy.ndarray' object has no attribute 'append'

In [ ]:
import numpy as np
print(f"D3QN_PER Final Evaluation Reward: {np.mean(d3qnPerFinalEvalReward).item()}")

NameError: name 'd3qnPerFinalEvalReward' is not defined

In [ ]:
plotQuantity(
    d3qnPerTrainRewardsList, 
    len(d3qnPerTrainRewardsList), 
    descriptionList = ["Episodes", "Train Rewards", "OBELIX Train Reward over Episodes using D3QN_PER"]    
)

NameError: name 'd3qnPerTrainRewardsList' is not defined

In [ ]:
plotQuantity(
    d3qnPerEvalRewardsList, 
    len(d3qnPerEvalRewardsList), 
    descriptionList = ["Episodes", "Eval Rewards", "OBELIX Evaluation Reward over Episodes using D3QN_PER"]    
)

NameError: name 'd3qnPerEvalRewardsList' is not defined

In [ ]:
plotQuantity(
    d3qnPerTrainTimeList, 
    len(d3qnPerTrainTimeList), 
    descriptionList = ["Episodes", "Train Time", "OBELIX Train Time over Episodes using D3QN_PER"]    
)

NameError: name 'd3qnPerTrainTimeList' is not defined

In [ ]:
plotQuantity(
    d3qnPerWallClockTimeList, 
    len(d3qnPerWallClockTimeList), 
    descriptionList = ["Episodes", "Wall Clock Time", "OBELIX WallClock Time over Episodes using D3QN_PER"]    
)

NameError: name 'd3qnPerWallClockTimeList' is not defined

In [ ]:
plotQuantity(
    d3qnPerTotalStepsList, 
    len(d3qnPerTotalStepsList), 
    descriptionList = ["Episodes", "Total Steps", "OBELIX Total Steps over Episodes using D3QN_PER"]    
)

NameError: name 'd3qnPerTotalStepsList' is not defined